In [12]:
@time using Pkg, OrdinaryDiffEq, CairoMakie, LaTeXStrings, Printf, Roots, GeometryBasics, Colors, ForwardDiff, Symbolics, UnPack

 33.175725 seconds (9.32 M allocations: 561.379 MiB, 5.47% gc time, 1.15% compilation time: 38% of which was recompilation)


In [29]:
function first_derivative_4th_order(f::Function, x::Float64, h::Float64)
    return (-f(x + 2h) + 8*f(x + h) - 8*f(x - h) + f(x - 2h)) / (12h)
end

function second_derivative_4th_order(f::Function, x::Float64, h::Float64)
    return (-f(x-2*h) + 16*f(x-h) - 30*f(x) + 16*f(x+h) - f(x+2*h)) / (12*h^2)
end

function third_derivative_4th_order(f::Function, x::Float64, h::Float64)
    return (-f(x-2*h) + 2*f(x-h) - 2*f(x+h) + f(x+2*h)) / (2*h^3)
end

third_derivative_4th_order (generic function with 1 method)

In [136]:
function classify_equilibria(f::Function; df::Function = x -> first_derivative_4th_order(f, x, 1e-3), a::Float64 = -10.0, b::Float64 = 10.0, 
                             df_tol::Float64 = 1e-10, f_tol::Float64 = 1e-10, Δx::Float64 = 1e-6, verbose::Bool = true)

    eq_all = find_zeros(f, a, b)   #Find the equilibria of the autonomous ODE ̇x = f(x) in the interval [a,b] 

    if isempty(eq_all) && verbose == true 
        println("No equilibria were found in the given interval.") 
    end 
    
    eq_stable = filter(x -> df(x) < -df_tol, eq_all) 
    eq_unstable = filter(x -> df(x) > df_tol, eq_all) 
    eq_marginal = filter(x -> abs(df(x)) ≤ df_tol, eq_all)    #Requires more analysis to determine the stability 

    eq_left_stable = Float64[]
    eq_right_stable = Float64[]

    for x_eq ∈ eq_marginal 

        d2f_eq = second_derivative_4th_order(f, x_eq, Δx)   #Second derivative of f at x = x_eq 
        d3f_eq = third_derivative_4th_order(f, x_eq, Δx)   #Third derivative of f at x = x_eq 

        #CHECK SECOND DERIVATIVE if f'(x_eq) is zero (approximately) 
        if d2f_eq > f_tol 
           push!(eq_left_stable, x_eq)
        elseif d2f_eq < -f_tol
           push!(eq_right_stable, x_eq)
        end

        #CHECK THIRD DERIVATIVE if f'(x_eq) and f''(x_eq) are both zero (approximately) 
        if abs(d2f_eq) ≤ df_tol                          
            if d3f_eq > df_tol
                push!(eq_unstable, x_eq)
            elseif d3f_eq < -df_tol
                push!(eq_stable, x_eq)
            end
        end
        
    end 

    eq_undetermined = setdiff(eq_marginal, union(eq_stable, eq_unstable, eq_left_stable, eq_right_stable)) 
    
    return (eq_all = eq_all, eq_stable = eq_stable, eq_unstable = eq_unstable, eq_left_stable = eq_left_stable, eq_right_stable = eq_right_stable, 
        eq_undetermined = eq_undetermined)

end

classify_equilibria (generic function with 1 method)

In [11]:
function phase_line(f::Function; df::Function = x -> ForwardDiff.derivative(f, x), a::Float64 = -10.0, b::Float64 = 10.0, 
                    df_tol::Float64 = 1e-5, f_tol::Float64 = 1e-20, Δx::Float64 = 1e-6, npoints::Int64 = 200, 
                    xmin = -2.0, xmax = 2.0, ymin = -2.0, ymax = 2.0, title = "Phase Line", xlabel = L"x", ylabel = L"\dot{x}", figscale = 1.0,
                    arrow_spacing = 0.25, exclusion_radius = 0.15, f_thresh = 0.15, arrow_scale = 1.0)

    MT = Makie.MathTeXEngine
    mt_fonts_dir = joinpath(dirname(pathof(MT)), "..", "assets", "fonts", "NewComputerModern")
    Makie.set_theme!(fonts = (regular = joinpath(mt_fonts_dir, "NewCM10-Regular.otf"), bold = joinpath(mt_fonts_dir, "NewCM10-Bold.otf"), 
                     italic = joinpath(mt_fonts_dir, "NewCM10-Italic.otf")))
    
    equilibria = classify_equilibria(f; df = df, a = a, b = b, df_tol = df_tol, f_tol = f_tol, Δx = Δx, verbose = false)
    @unpack eq_all, eq_stable, eq_unstable, eq_left_stable, eq_right_stable = equilibria 
    
    markersize = 10 * arrow_scale
    shaftlength = 6 * arrow_scale
    shaftwidth = 1.5 * arrow_scale

    shaftcolor = :blue 
    linewidth = 1
       
    fig = Figure(size = figscale .* (400, 300))
    ax = Axis(fig[1,1], limits = (xmin, xmax, ymin, ymax), title = title, xlabel = xlabel, ylabel = ylabel)
    hlines!(ax, 0, color = :black, linewidth = 0.5)    #Plot horizontal line at y = 0 


    #PLOT f(x) vs x 
    x = collect(LinRange(xmin, xmax, npoints))
    y = f.(x) 
    lines!(ax, x, y, linewidth = linewidth)


    #PLOT THE STABLE EQUILIBRIA
    if length(eq_stable) > 0
        scatter!(ax, eq_stable, zero(eq_stable), marker = '●', color = :black, markersize = markersize)
    end

    #PLOT THE UNSTABLE EQUILIBRIA
    if length(eq_unstable) > 0
        scatter!(ax, eq_unstable, zero(eq_unstable), color = :white, strokecolor = :black, strokewidth = 1.5)
       
    end

    
    #PLOT THE LEFT-STABLE EQUILIBRIA
    if length(eq_left_stable) > 0
        scatter!(ax, eq_left_stable, zero(eq_left_stable), marker = '◐',  markersize = markersize, color = :black)
    end

    #PLOT THE RIGHT-STABLE EQUILIBRIA
    if length(eq_right_stable) > 0
        scatter!(ax, eq_right_stable, zero(eq_right_stable), marker = '◑',  markersize = markersize, color = :black)
    end

    
    #SET ARROW POSITIONS (avoiding regions near equilibria) 
    #arrow_spacing = 0.25
    #exclusion_radius = 0.15       #Don't place arrows this close to fixed points
    
    arrow_positions = Float64[]
    x_current = xmin + arrow_spacing / 2
    
    while x_current < xmax - arrow_spacing / 2
        # Check if we're too close to any fixed point
        too_close = any(abs(x_current - x_eq) < exclusion_radius for x_eq in eq_all)
        
        if !too_close && abs(f(x_current)) > f_thresh  # Significant flow
            push!(arrow_positions, x_current)
        end
        x_current += arrow_spacing
    end
    
    if length(arrow_positions) > 0
        arrow_y = zeros(length(arrow_positions))
        flow_dirs = [f(x) for x in arrow_positions]
        arrow_u = sign.(flow_dirs) .* 0.12 * arrow_scale
        arrow_v = zeros(length(arrow_positions))
        arrows2d!(ax, arrow_positions, arrow_y, arrow_u, arrow_v, shaftlength = shaftlength, shaftcolor = shaftcolor, shaftwidth = shaftwidth)
    end
    
        
    return fig
    
end

LoadError: LoadError: UndefVarError: `@L_str` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at In[11]:3

In [139]:
function bifurcation_diagram(F::Function; λ_vals::Vector{Float64}, xmin::Float64, xmax::Float64, title = nothing, λ_crit = nothing)

    points_stable = Tuple{Float64, Float64}[]
    points_unstable = Tuple{Float64, Float64}[]
    points_left_stable = Tuple{Float64, Float64}[]
    points_right_stable = Tuple{Float64, Float64}[]
    
    for λ in λ_vals

        f = x -> F(x,λ)
        equilibria = classify_equilibria(f; df = x -> first_derivative_4th_order(f, x, 1e-3), a = xmin, b = xmax, verbose = false)        
        @unpack eq_stable, eq_unstable, eq_left_stable, eq_right_stable = equilibria
        
        append!(points_stable, [(λ, x) for x in eq_stable])
        append!(points_unstable, [(λ, x) for x in eq_unstable])
        append!(points_left_stable, [(λ, x) for x in eq_left_stable])
        append!(points_right_stable, [(λ, x) for x in eq_right_stable])        
    end

    λ_min, λ_max = extrema(λ_vals)
    
    fig = Figure()
    ax = Axis(fig[1,1], limits = (λ_min, λ_max, xmin, xmax), xlabel = L"λ", ylabel = L"x^{*}", title = title)
    scatter!(ax, points_stable, color = :blue, markersize = 3, label = "stable")
    scatter!(ax, points_unstable, color = :red, markersize = 3, label = "unstable")
    scatter!(ax, points_left_stable, color = :purple, markersize = 3, label = "left stable")
    scatter!(ax, points_right_stable, color = :cyan, markersize = 3, label = "right stable")

    if !isnothing(λ_crit)
        vlines!(ax, λ_crit, color = :black, linestyle = :dash, linewidth = 1)
    end
    
    points_semistable = vcat(points_left_stable, points_right_stable)
    println(points_semistable)
    

    #legend = Legend(fig, ax, framevisible = true)


    legend = Legend(fig, ax, framevisible = true;
                    labelsize = 15,        # font size
                    patchsize = (30, 20),  # marker box size
                    markersize = 300        # size of scatter markers in legend
    )

    fig[1,2] = legend 
    return fig
end 

bifurcation_diagram (generic function with 2 methods)

In [1]:
function plot_phase_portrait(;μ, ntraj, t0 = 0.0, tf = 1.0)

    #ntraj : the number of trajectories to plot 

    fixed_points = find_zeros(x -> f(x,μ), -5.0, 5.0)
  
    if length(fixed_points) > 0
        fp_min, fp_max = extrema(fixed_points) 
        x0_min = fp_min - 0.2*abs(fp_min)
        x0_max = fp_max + 0.2*abs(fp_max)
        x0_vals = collect(LinRange(x0_min, x0_max, ntraj))
    else
        x0_min = -2.5
        x0_max = 2.5
        x0_vals = collect(LinRange(x0_min, x0_max, 4)) 
    end
    
    npoints = 250
    tvals = LinRange(t0, tf, npoints)
    
    fig = Figure()

    #limits for the y-axis of the plot
    ymin = x0_min - 0.25*abs(x0_min)   
    ymax = x0_max + 0.25*abs(x0_max)
    
    ax = Axis(fig[1,1], xlabel = L"t", ylabel = L"x", title = L"μ = %$μ", limits = (t0, tf, ymin, ymax), titlesize = 24,
             xlabelsize = 22, ylabelsize = 22)

    #Plot horizontal lines for the fixed points 
    if length(fixed_points) > 0
        hlines!(ax, fixed_points, color = :black, linewidth = 1)  
    end
    
    for x0 ∈ x0_vals
        prob = ODEProblem((x,μ,t) -> f(x,μ), x0, (t0, tf), μ)
        sol = solve(prob, Tsit5(), reltol = 1e-3, abstol = 1e-3, saveat = tvals)
        lines!(ax, sol.t, sol.u)
    end

    return fig
    
end

LoadError: LoadError: UndefVarError: `@L_str` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at In[1]:27